## Import and load ##

In [1]:
# cell 1: imports & basic config
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# paths
root = Path('.')  # notebook is in TREES/
images_dir = root / "images"

Device: cpu


### Compute classes and split dataset

In [2]:
basic_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
full_ds = datasets.ImageFolder(str(images_dir), transform=basic_tf)
class_to_idx = full_ds.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
num_classes = len(class_to_idx)
print("Found classes:", num_classes, idx_to_class)

Found classes: 5 {0: 'koivu', 1: 'kuusi', 2: 'lehmus', 3: 'pihlaja', 4: 'vaahtera'}


In [3]:
labels = [s[1] for s in full_ds.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=42)

### Train and test

In [4]:
# Training and test transforms
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

from torch.utils.data import Subset

train_ds = Subset(datasets.ImageFolder(str(images_dir), transform=train_tf), train_idx)
test_ds  = Subset(datasets.ImageFolder(str(images_dir), transform=test_tf),  val_idx)  # rename val -> test

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Train/test sizes:", len(train_ds), len(test_ds))


Train/test sizes: 108 28


In [5]:
model = models.resnet50(pretrained=True)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)  # adjust for your number of species
model = model.to(device)


/opt/anaconda3/envs/SolutionsInPR/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/SolutionsInPR/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/iirokaki/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:05<00:00, 18.4MB/s]


In [6]:
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)


In [8]:
def evaluate(model, loader, device):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            preds = out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
    return correct / total


In [9]:
n_epochs = 25

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}/{n_epochs} - Loss: {avg_loss:.4f} - Test Accuracy: {test_acc:.4f}")


KeyboardInterrupt: 

In [ ]:
# unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.001)
n_epochs_ft = 15

for epoch in range(n_epochs_ft):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)
    test_acc = evaluate(model, test_loader, device)
    print(f"FT Epoch {epoch+1}/{n_epochs_ft} - Loss: {avg_loss:.4f} - Test Accuracy: {test_acc:.4f}")

torch.save(model.state_dict(), "models/resnet50_finetunedV3.pth")


c:\Users\rawil\anaconda3\envs\SoPa\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


FT Epoch 1/15 - Loss: 0.8918 - Test Accuracy: 0.1786
FT Epoch 2/15 - Loss: 1.2747 - Test Accuracy: 0.1786
FT Epoch 3/15 - Loss: 1.6514 - Test Accuracy: 0.1786
FT Epoch 4/15 - Loss: 0.9215 - Test Accuracy: 0.6429


In [ ]:
def predict_image(img_path, model, idx_to_class, device):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    x = test_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(x)
        probs = torch.nn.functional.softmax(out, dim=1)
        p, pred = torch.max(probs, dim=1)
    return idx_to_class[int(pred.item())], float(p.item())

# example
print(predict_image("images/pihlaja/pihlaja9.jpg", model, idx_to_class, device))


('pihlaja', 0.8389784097671509)
